# Improvement results and model decision

Compare development trials at the prespecified objective; do not choose a deployment checkpoint from fold_2.

Run from the existing `leukemia-yolo` Conda kernel. Source datasets and earlier results are read only.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if ROOT.name != 'leukemia-yolo': ROOT = ROOT.parent
assert (ROOT / 'config.local.yaml').is_file()
print(sys.executable, ROOT)

In [ ]:
import json
base=ROOT/'outputs/classifier/improvements'
rows=[]
for arm in ['control','augmentation_trial','class_weight_trial']:
    runs=sorted((base/arm).glob('trial_*/completed.json'))
    if runs:
        run=runs[-1].parent
        t=json.loads((run/'threshold_config.json').read_text())
        h=pd.read_csv(run/'training_history.csv')
        best=h[h.epoch==t['epoch']].iloc[0]
        rows.append({'arm':arm,'epoch':t['epoch'],'threshold':t['threshold'],'eligible':t['eligible'],'specificity':best.val_specificity,'sensitivity':best.val_sensitivity,'macro_f1':best.val_macro_f1,'roc_auc':best.val_roc_auc})
display(pd.DataFrame(rows))

In [ ]:
if rows:
    data=pd.DataFrame(rows)
    data.plot.bar(x='arm',y=['specificity','sensitivity','macro_f1'],ylim=(0,1),figsize=(9,4));plt.show()
print('Current full model remains the dashboard default until an independently justified replacement is approved.')

In [ ]:
paper=ROOT/'outputs/audit/conference_status.md'
print(paper.read_text(encoding='utf-8')[-1800:])